# This is a demo of BERTopic

(see https://maartengr.github.io/BERTopic/getting_started/quickstart/quickstart.html)

Note that this notebook exists just for convenience reasons and to get started!

Read the original tutorial to get complete information (in particular, also about different transformer models, for example for other languages than English)

In [ ]:
#uncomment the next line to install the required packages:
# %pip install bertopic sentence-transformers transformers umap-learn HDBSCAN hf_xet nbformat 

In [ ]:
from bertopic import BERTopic
from sklearn.datasets import fetch_20newsgroups

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer

# Load the model and tokenizer
model_name = 'jegorkitskerkin/robbert-v2-dutch-base-mqa-finetuned'
model = SentenceTransformer(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Your sentences
sentences = ["Dit is een voorbeeld zin", "Elke zin wordt omgezet"]

# Get tokens
tokens = tokenizer(sentences, padding=True, truncation=True, return_tensors="pt")

# Get embeddings
embeddings = model.encode(sentences)

# Now you have both tokens and embeddings
print("Tokens:", tokens[0].tokens)
print("Embeddings shape:", embeddings.shape)

## 1. Load Data

In [ ]:
docs = fetch_20newsgroups(subset='all',  remove=('headers', 'footers', 'quotes'))['data']

print("Number of documents:", len(docs))
print("First document:", docs[0])

## 2. Train a BERTopic model: 

The following elements comprise the BERTopic stack:

<img src="../images/overview_bert.png" alt="Default Settings of BERTopic" width="300" height="200">

https://maartengr.github.io/BERTopic/algorithm/algorithm.html#visual-overview

1. create embeddings from documents (with Sentence Transformers)
2. reduce the dimensionality of the embeddings (with UMAP)
3. cluster the reduced embeddings (with HDBSCAN)
4. find words that describe the similarity found within the clusters to describe their topic (with c-TF-IDF)
5. optional refine these descriptions for easier interpretation (advanced, including LLMs)


**NOTE: simply running BERTopic will automatically execute all of these steps!**



In [ ]:
#initialize the model:
from umap import UMAP
umap_model = UMAP(n_neighbors=15, n_components=5, 
                  min_dist=0.0, metric='cosine', random_state=42)


topic_model = BERTopic(embedding_model="paraphrase-MiniLM-L6-v2", umap_model=umap_model)

#fit the model to the texts:
#NOTE: this may take a while on a CPU! (5-6 minutes on mine)
topics, probs = topic_model.fit_transform(docs)

#NOTE: notice how we do not perform any preprocessing on the documents!

## 3. Inspect the results: 

In [ ]:
#let's see the topics:
topic_model.get_topic_info()

In [ ]:
#let's get the topN (c-TF-IDF based) words for a specific topic:
topic_model.get_topic(1)

In [ ]:
#the information we get for each document:
topic_model.get_document_info(docs)

## 4. change the topic labels

In [ ]:
# automatically, based on top3 words:

topic_labels = topic_model.generate_topic_labels(nr_words=3,
                                                 topic_prefix=False,
                                                 word_length=10,
                                                 separator=", ")

topic_model.set_topic_labels(topic_labels)

## 5. Visualize the results:

In [ ]:
#how the topics are distributed:
#NOTE: you can hover / select topics in the plot!
topic_model.visualize_topics()

In [ ]:
topic_model.visualize_barchart(top_n_topics=10)

In [ ]:
#visualize the hierarchy of topics:
topic_model.visualize_hierarchy()

In [ ]:
# Compute embeddings for your documents (which were used to fit the model above)
embeddings = topic_model.embedding_model.embed(docs)

# Reduce dimensionality to 2D for visualization (now look at the simplified 2D representation of each document for illustration (rather than the 5D used as input for the clustering above))
umap_model_2 = UMAP(n_neighbors=15, n_components=2, min_dist=0.0, metric='cosine', random_state=42)
reduced_embeddings = umap_model_2.fit_transform(embeddings)

# Now visualize how the documents are distributed in the 2D space (with topic labels per document based on the BERTopic model)
topic_model.visualize_documents(docs, reduced_embeddings=reduced_embeddings)

In [ ]:
topic_model.visualize_heatmap()

## 6. Advanced: building your own BERTopic stack:

### Modularity: Choosing Your Preferred BERT Flavor
Customize the foundational elements of BERTopic to create the optimal model for your specific use case.

<img src="../images/modularity_bert.png" alt="Default Settings of BERTopic" width="600" height="600">

https://maartengr.github.io/BERTopic/algorithm/algorithm.html#visual-overview


In [ ]:
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic.representation import KeyBERTInspired

# Step 1 - Extract embeddings
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Step 2 - Reduce dimensionality
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine')

# Step 3 - Cluster reduced embeddings
hdbscan_model = HDBSCAN(min_cluster_size=15, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

# Step 4 - Tokenize topics
vectorizer_model = CountVectorizer(stop_words="english")

# Step 5 - Create topic representation
ctfidf_model = ClassTfidfTransformer()

# Step 6 - (Optional) Fine-tune topic representations with 
# a `bertopic.representation` model
representation_model = KeyBERTInspired()

# All steps together
topic_model = BERTopic(
  embedding_model=embedding_model,          # Step 1 - Extract embeddings
  umap_model=umap_model,                    # Step 2 - Reduce dimensionality
  hdbscan_model=hdbscan_model,              # Step 3 - Cluster reduced embeddings
  vectorizer_model=vectorizer_model,        # Step 4 - Tokenize topics
  ctfidf_model=ctfidf_model,                # Step 5 - Extract topic words
  representation_model=representation_model # Step 6 - (Optional) Fine-tune topic represenations
)


In [ ]:
topics, probs = topic_model.fit_transform(docs)

In [ ]:
embeddings = embedding_model.encode(docs, show_progress_bar=False)

topic_model.visualize_documents(docs, embeddings=embeddings)

In [ ]:
# Reduce dimensionality of embeddings, this step is optional but much faster to perform iteratively:
reduced_embeddings = UMAP(n_neighbors=10, n_components=2, min_dist=0.0, metric='cosine').fit_transform(embeddings)
topic_model.visualize_documents(docs, reduced_embeddings=reduced_embeddings)

### Changing pieces of the stack: PCA, K-means and TFidF

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Step 1 - Extract embeddings
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Step 2 - Reduce dimensionality
dim_model = PCA(n_components=5)

# Step 3 - Cluster reduced embeddings
cluster_model = KMeans(n_clusters=50)

# Step 4 - Tokenize topics
vectorizer_model = TfidfVectorizer(stop_words="english")

# Step 5 - Create topic representation
ctfidf_model = ClassTfidfTransformer()

# Step 6 - (Optional) Fine-tune topic representations with 
# a `bertopic.representation` model
representation_model = KeyBERTInspired()

# All steps together
topic_model = BERTopic(
  embedding_model=embedding_model,          # Step 1 - Extract embeddings
  umap_model=dim_model,                    # Step 2 - Reduce dimensionality
  hdbscan_model=cluster_model,              # Step 3 - Cluster reduced embeddings
  vectorizer_model=vectorizer_model,        # Step 4 - Tokenize topics
  ctfidf_model=ctfidf_model,                # Step 5 - Extract topic words
  representation_model=representation_model # Step 6 - (Optional) Fine-tune topic represenations
)

In [ ]:
topics, probs = topic_model.fit_transform(docs)

In [ ]:
#or even more crudely, use a Bag of Words (BoW) vectorizer like CountVectorizer() rather than an LLM like "all-MiniLM-L6-v2" within the BERTopic pipeline like in the k-means example this morning:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans


# Step 1: Create TF-IDF matrix (these are your "embeddings")
vectorizer = TfidfVectorizer(stop_words="english")
X_tfidf = vectorizer.fit_transform(docs)

# Step 2: Cluster directly on TF-IDF features
cluster_model = KMeans(n_clusters=50)
clusters = cluster_model.fit_predict(X_tfidf)

# Step 3: Use BERTopic with precomputed "embeddings" (TF-IDF matrix)
topic_model_tfidf = BERTopic(
    embedding_model=None,                    # No dense embeddings, use X_tfidf
    umap_model=None,                         # No dimensionality reduction
    hdbscan_model=cluster_model,             # KMeans for clustering
    vectorizer_model=vectorizer,             # TF-IDF vectorizer
    ctfidf_model=ClassTfidfTransformer(),    # c-TF-IDF for topic words
    representation_model=None                 # Optional fine-tuning (keybertinspired() expects a dense embedding input, so we must skip it here)
)

topics, probs = topic_model_tfidf.fit_transform(docs, X_tfidf) #note we pass X_tfidf here as embeddings directly

In [ ]:
# Reduce TF-IDF features to 2D for visualization
umap_model = UMAP(n_neighbors=15, n_components=2, min_dist=0.0, metric='cosine', random_state=42)
reduced_tfidf = umap_model.fit_transform(X_tfidf)


topic_model_tfidf.visualize_documents(docs, reduced_embeddings=reduced_tfidf)